# 2. 문서 파이프라인 & Naive RAG

## 학습 목표

| 파트 | 주제 | 핵심 내용 |
|------|------|----------|
| **Part 1** | 문서 로딩·분할·임베딩 파이프라인 | PDF, Excel, Word 등 다양한 포맷의 문서를 로드하고, 청크로 분할한 뒤 임베딩 벡터로 변환하는 전체 흐름을 설계한다 |
| **Part 2** | 벡터 검색 & 검색 Tool 래핑 | 벡터스토어에서 의미론적 검색을 수행하고, Retriever를 `@tool`로 래핑하여 **Agent가 자율 호출할 수 있는 검색 도구**로 변환한다. 미들웨어로 Agent 행동을 제어하는 **하네스** 개념을 학습한다 |
| **Part 3** | Agent 기반 Naive RAG & End-to-End Q&A | Agent가 검색 Tool을 자율 호출하는 **Naive RAG** 아키텍처를 구현하고, 검색 품질과 프롬프트를 튜닝한다 |

### 전체 파이프라인

```
문서(PDF/Excel/Word) → 로딩 → 텍스트 분할 → 임베딩 → 벡터스토어 → 검색 Tool 래핑 → Agent 자율 호출 → 답변
```

> **이 Lab의 핵심:** 문서를 Agent의 검색 Tool로 변환하고, 미들웨어로 Agent 행동을 제어하는 방법을 학습합니다. Naive RAG는 가장 기본적인 RAG 아키텍처로, Agent가 단일 검색 도구를 호출하여 정보를 조회하고 답변을 생성합니다. 이 Lab에서는 Naive RAG의 구현과 한계를 모두 경험한 뒤, 다음 Lab의 **Advanced RAG**로 전환하는 근거를 확보한다.



## 0) 환경 설정

필요한 패키지를 설치하고, API 키를 로드한다.

| 패키지 | 역할 |
|--------|------|
| `langchain-openai` | OpenAI LLM·임베딩 연동 |
| `langchain-chroma chromadb` | ChromaDB 벡터스토어 |
| `langchain-text-splitters` | 텍스트 분할기 |
| `langchain-community` | PDF·Excel·Word 문서 로더 |
| `pypdf` | PDF 파싱 백엔드 |
| `openpyxl` | Excel 파싱 백엔드 |
| `python-docx` | Word 파싱 백엔드 |
| `python-dotenv` | `.env` 파일에서 환경 변수 로드 |


In [1]:
# [2-1] : 패키지 설치
# [핵심] langchain-chroma chromadb — ChromaDB 벡터스토어, langchain-text-splitters — 청크 분할
# [주의] pypdf·openpyxl·python-docx 는 각 포맷 로더의 파싱 백엔드 — 누락 시 ImportError
!uv pip install -q langchain-openai langchain-chroma chromadb langchain-text-splitters langchain-community pypdf openpyxl python-docx python-dotenv unstructured networkx msoffcrypto-tool docx2txt

In [2]:
# [2-2] : 환경 변수 로드 & API 키 확인
# [핵심] .env 파일에서 OPENAI_API_KEY 를 읽어 환경 변수로 설정
# [주의] assert 실패 시 .env 파일에 OPENAI_API_KEY=sk-... 를 추가할 것
from dotenv import load_dotenv
import os

load_dotenv(override=True)  # .env 파일 로드 — override=True 로 기존 값 덮어쓰기

assert os.environ.get("OPENAI_API_KEY"), "⚠️ OPENAI_API_KEY를 .env 파일에 설정하세요"
print("API 키 로드 완료")

API 키 로드 완료


In [3]:
# [2-3] : LLM & 임베딩 모델 초기화
# [핵심] gpt-4.1 — 생성 모델, text-embedding-3-small — 임베딩 모델
# [주의] 임베딩 모델은 벡터 차원이 고정됨 — 스토어 생성 후 모델 변경 불가
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

llm = ChatOpenAI(model="gpt-4.1", temperature=0)       # 생성 모델
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")   # 임베딩 모델 (1536차원)

print(f"LLM: {llm.model_name}")
print(f"Embeddings: {embeddings.model}")

LLM: gpt-4.1
Embeddings: text-embedding-3-small


In [4]:
llm.invoke("Hello, world!")  # 모델 테스트 — 오류 시 API 키 및 모델 이름 확인

AIMessage(content='Hello, world! 👋 How can I help you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 13, 'prompt_tokens': 11, 'total_tokens': 24, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-2025-04-14', 'system_fingerprint': 'fp_0b1cdfeeef', 'id': 'chatcmpl-DrD4As9WFADe5drX8gjtzo0N8uDbl', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019ece15-9626-76a0-937a-f23c0a177c89-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 11, 'output_tokens': 13, 'total_tokens': 24, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [5]:
embeddings.embed_query("점심메뉴 추천해줘")  # 임베딩 테스트 — 오류 시 API 키 및 모델 이름 확인

[-0.0151519775390625,
 0.0025844573974609375,
 -0.057159423828125,
 0.01409149169921875,
 -0.00618743896484375,
 -0.0216217041015625,
 -0.06475830078125,
 0.0305328369140625,
 -0.008056640625,
 -0.01062774658203125,
 -0.0018482208251953125,
 0.02105712890625,
 -0.006237030029296875,
 0.01617431640625,
 0.01261138916015625,
 -0.00798797607421875,
 -0.0489501953125,
 -0.02099609375,
 0.036956787109375,
 0.0738525390625,
 0.0156097412109375,
 0.005283355712890625,
 0.00809478759765625,
 0.0251617431640625,
 0.04046630859375,
 0.018402099609375,
 -0.0174407958984375,
 0.017364501953125,
 -0.00562286376953125,
 0.0008187294006347656,
 -0.0016307830810546875,
 -0.0203704833984375,
 0.017608642578125,
 -0.03533935546875,
 -0.0209808349609375,
 0.00383758544921875,
 0.016448974609375,
 1.2874603271484375e-05,
 0.0003533363342285156,
 -0.0018854141235351562,
 0.0026226043701171875,
 0.01690673828125,
 0.0604248046875,
 0.007801055908203125,
 0.01422119140625,
 0.045013427734375,
 -0.04104614257

---

## 1) 문서 로딩 — PDF / Excel / Word

RAG 파이프라인의 첫 단계는 **원본 문서를 `Document` 객체로 변환**하는 것이다. LangChain은 포맷별 로더를 제공한다.

| 포맷 | 로더 | 파싱 백엔드 |
|------|------|------------|
| **PDF** | `PyPDFLoader` | `pypdf` |
| **Excel** | `UnstructuredExcelLoader` | `openpyxl` |
| **Word** | `Docx2txtLoader` | `python-docx` |

`Document` 객체 구조:
- **`page_content`** — 텍스트 본문
- **`metadata`** — 출처, 페이지 번호 등 부가 정보

> 실무에서는 문서 포맷이 혼재하므로, 포맷 감지 → 적절한 로더 선택 → 통합 `Document` 리스트 생성 패턴이 필수다.

In [24]:
# [2-4] : 포맷별 문서 로딩
# [핵심] 각 로더는 Document(page_content, metadata) 리스트를 반환
# [주의] PDF 한글 깨짐 시 PyMuPDFLoader 대안 사용, Excel 은 시트별 Document 1개
from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader, UnstructuredExcelLoader

# --- 경로 지정 ---
pdf_path = "./sample_docs/ai_agent_overview.pdf"
excel_path = "./sample_docs/process_data.xlsx"
docx_path = "./sample_docs/quality_guide.docx"

In [25]:
# --- PDF 로딩 ---
pdf_loader = PyPDFLoader(pdf_path)
pdf_docs = pdf_loader.load()  # 페이지별 Document 생성
print(f"[PDF] {len(pdf_docs)}개 Document 로드")

pdf_docs[0].page_content


[PDF] 1개 Document 로드


'AI Agent Overview\nAI agents use LLMs to understand user goals and perform tasks.\nCore components: LLM, tools, memory.\nRepresentative patterns: ReAct, Plan-and-Execute, Reflection, RAG.'

In [20]:
# --- Excel 로딩 ---
excel_loader = UnstructuredExcelLoader(excel_path)
excel_docs = excel_loader.load()  # 시트별 Document 생성
print(f"[Excel] {len(excel_docs)}개 Document 로드")
excel_docs[0].page_content

[Excel] 1개 Document 로드


'공정단계 설비명 주요파라미터 정상범위 비고 노광 ASML EUV 초점거리(nm) 13.2-13.6 EUV 리소그래피 식각 Lam Research 식각속도(nm/min) 50-80 건식 식각 증착 AMAT CVD 증착온도(C) 350-450 화학기상증착 세정 DNS 세정시간(sec) 30-60 SC-1 세정 검사 KLA 결함밀도(/cm2) 0-0.1 웨이퍼 검사 패키징 ASE 본딩온도(C) 150-200 와이어 본딩'

In [21]:
# --- Word 로딩 ---
docx_loader = Docx2txtLoader(docx_path)
docx_docs = docx_loader.load()
print(f"[Word] {len(docx_docs)}개 Document 로드")
docx_docs[0].page_content

[Word] 1개 Document 로드


'반도체 품질 관리 가이드\n\n반도체 제조 공정에서 품질 관리는 수율(Yield)을 결정하는 핵심 요소이다. 웨이퍼 단위의 결함 검사, 전기적 특성 테스트, 신뢰성 시험을 통해 제품 품질을 보증한다.\n\n주요 검사 항목\n\n결함 밀도 검사 (Defect Density)\n\n전기적 특성 테스트 (WAT)\n\n신뢰성 시험 (HTOL, TC)\n\n외관 검사 (Visual Inspection)\n\n품질 개선 프로세스\n\nPDCA(Plan-Do-Check-Act) 사이클을 기반으로 지속적 개선을 수행한다. 데이터 기반 의사결정을 위해 SPC(통계적 공정 관리) 차트를 활용하며, 이상 발생 시 8D 리포트를 작성하여 근본 원인을 분석한다.'

In [22]:
# --- 전체 통합 ---
all_docs = pdf_docs + excel_docs + docx_docs
print(f"\n전체 로드 문서: {len(all_docs)}개")


전체 로드 문서: 3개


In [23]:
all_docs

[Document(metadata={'producer': 'Matplotlib pdf backend v3.10.8', 'creator': 'Matplotlib v3.10.8, https://matplotlib.org', 'creationdate': '2026-04-23T11:49:08+09:00', 'source': './sample_docs/ai_agent_overview.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}, page_content='AI Agent Overview\nAI agents use LLMs to understand user goals and perform tasks.\nCore components: LLM, tools, memory.\nRepresentative patterns: ReAct, Plan-and-Execute, Reflection, RAG.'),
 Document(metadata={'source': './sample_docs/process_data.xlsx'}, page_content='공정단계 설비명 주요파라미터 정상범위 비고 노광 ASML EUV 초점거리(nm) 13.2-13.6 EUV 리소그래피 식각 Lam Research 식각속도(nm/min) 50-80 건식 식각 증착 AMAT CVD 증착온도(C) 350-450 화학기상증착 세정 DNS 세정시간(sec) 30-60 SC-1 세정 검사 KLA 결함밀도(/cm2) 0-0.1 웨이퍼 검사 패키징 ASE 본딩온도(C) 150-200 와이어 본딩'),
 Document(metadata={'source': './sample_docs/quality_guide.docx'}, page_content='반도체 품질 관리 가이드\n\n반도체 제조 공정에서 품질 관리는 수율(Yield)을 결정하는 핵심 요소이다. 웨이퍼 단위의 결함 검사, 전기적 특성 테스트, 신뢰성 시험을 통해 제품 품질을 보증한다.\n\n주요 검사 항목\n\n결함 밀

In [26]:
# --- 메타데이터 표준화 (Chroma 메타데이터 호환) ---
# PDF/Excel/Word 로더가 각각 다른 메타데이터를 추가하므로, 공통 필드만 유지
for doc in all_docs:
    doc.metadata = {"source": doc.metadata.get("source", "unknown")}
all_docs

[Document(metadata={'source': './sample_docs/ai_agent_overview.pdf'}, page_content='AI Agent Overview\nAI agents use LLMs to understand user goals and perform tasks.\nCore components: LLM, tools, memory.\nRepresentative patterns: ReAct, Plan-and-Execute, Reflection, RAG.'),
 Document(metadata={'source': './sample_docs/process_data.xlsx'}, page_content='공정단계 설비명 주요파라미터 정상범위 비고 노광 ASML EUV 초점거리(nm) 13.2-13.6 EUV 리소그래피 식각 Lam Research 식각속도(nm/min) 50-80 건식 식각 증착 AMAT CVD 증착온도(C) 350-450 화학기상증착 세정 DNS 세정시간(sec) 30-60 SC-1 세정 검사 KLA 결함밀도(/cm2) 0-0.1 웨이퍼 검사 패키징 ASE 본딩온도(C) 150-200 와이어 본딩'),
 Document(metadata={'source': './sample_docs/quality_guide.docx'}, page_content='반도체 품질 관리 가이드\n\n반도체 제조 공정에서 품질 관리는 수율(Yield)을 결정하는 핵심 요소이다. 웨이퍼 단위의 결함 검사, 전기적 특성 테스트, 신뢰성 시험을 통해 제품 품질을 보증한다.\n\n주요 검사 항목\n\n결함 밀도 검사 (Defect Density)\n\n전기적 특성 테스트 (WAT)\n\n신뢰성 시험 (HTOL, TC)\n\n외관 검사 (Visual Inspection)\n\n품질 개선 프로세스\n\nPDCA(Plan-Do-Check-Act) 사이클을 기반으로 지속적 개선을 수행한다. 데이터 기반 의사결정을 위해 SPC(통계적 공정 관리) 차트를 활용하며

---

## 2) 텍스트 분할 전략

문서를 통째로 임베딩하면 **의미가 희석**되고, LLM 컨텍스트 창을 낭비한다. 따라서 적절한 크기의 **청크(chunk)** 로 분할해야 한다.

### `RecursiveCharacterTextSplitter`

가장 범용적인 분할기로, **구분자 우선순위**를 따라 자연스러운 경계에서 분할한다.

```
분할 순서: "\n\n" → "\n" → " " → ""  (단락 → 줄 → 단어 → 글자)
```

| 파라미터 | 설명 | 권장값 |
|----------|------|--------|
| **`chunk_size`** | 청크 최대 글자 수 | 500~1000 |
| **`chunk_overlap`** | 인접 청크 간 겹침 글자 수 | chunk_size의 10~20% |

> **`chunk_overlap`** 이 중요한 이유 — 문장이 청크 경계에서 잘리면 의미가 손실된다. 겹침 구간이 이를 완화한다.

In [34]:
# [2-5] : 텍스트 분할 — RecursiveCharacterTextSplitter
# [핵심] chunk_size=500, chunk_overlap=50 — 의미 단위 보존과 검색 정밀도의 균형
# [주의] chunk_size 가 너무 작으면 문맥 손실, 너무 크면 검색 정밀도 하락
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)

In [35]:
# 전체 문서 분할
splits = text_splitter.create_documents([doc.page_content for doc in all_docs], metadatas=[doc.metadata for doc in all_docs])

print(f"원본 문서: {len(all_docs)}개 → 분할 청크: {len(splits)}개")
print(f"chunk_size=500, chunk_overlap=50\n")
splits

원본 문서: 3개 → 분할 청크: 3개
chunk_size=500, chunk_overlap=50



[Document(metadata={'source': './sample_docs/ai_agent_overview.pdf'}, page_content='AI Agent Overview\nAI agents use LLMs to understand user goals and perform tasks.\nCore components: LLM, tools, memory.\nRepresentative patterns: ReAct, Plan-and-Execute, Reflection, RAG.'),
 Document(metadata={'source': './sample_docs/process_data.xlsx'}, page_content='공정단계 설비명 주요파라미터 정상범위 비고 노광 ASML EUV 초점거리(nm) 13.2-13.6 EUV 리소그래피 식각 Lam Research 식각속도(nm/min) 50-80 건식 식각 증착 AMAT CVD 증착온도(C) 350-450 화학기상증착 세정 DNS 세정시간(sec) 30-60 SC-1 세정 검사 KLA 결함밀도(/cm2) 0-0.1 웨이퍼 검사 패키징 ASE 본딩온도(C) 150-200 와이어 본딩'),
 Document(metadata={'source': './sample_docs/quality_guide.docx'}, page_content='반도체 품질 관리 가이드\n\n반도체 제조 공정에서 품질 관리는 수율(Yield)을 결정하는 핵심 요소이다. 웨이퍼 단위의 결함 검사, 전기적 특성 테스트, 신뢰성 시험을 통해 제품 품질을 보증한다.\n\n주요 검사 항목\n\n결함 밀도 검사 (Defect Density)\n\n전기적 특성 테스트 (WAT)\n\n신뢰성 시험 (HTOL, TC)\n\n외관 검사 (Visual Inspection)\n\n품질 개선 프로세스\n\nPDCA(Plan-Do-Check-Act) 사이클을 기반으로 지속적 개선을 수행한다. 데이터 기반 의사결정을 위해 SPC(통계적 공정 관리) 차트를 활용하며

In [32]:
# 분할 결과 확인
for i, chunk in enumerate(splits):
    source = chunk.metadata.get("source", "unknown")
    print(f"  [{i}] ({len(chunk.page_content):3d}자) source={source}")
    print(f"       {chunk.page_content[:70]}...")
    print()

  [0] (184자) source=./sample_docs/ai_agent_overview.pdf
       AI Agent Overview
AI agents use LLMs to understand user goals and perf...

  [1] (199자) source=./sample_docs/process_data.xlsx
       공정단계 설비명 주요파라미터 정상범위 비고 노광 ASML EUV 초점거리(nm) 13.2-13.6 EUV 리소그래피 식각 La...

  [2] ( 75자) source=./sample_docs/process_data.xlsx
       30-60 SC-1 세정 검사 KLA 결함밀도(/cm2) 0-0.1 웨이퍼 검사 패키징 ASE 본딩온도(C) 150-200 와...

  [3] (183자) source=./sample_docs/quality_guide.docx
       반도체 품질 관리 가이드

반도체 제조 공정에서 품질 관리는 수율(Yield)을 결정하는 핵심 요소이다. 웨이퍼 단위의 결함 ...

  [4] (198자) source=./sample_docs/quality_guide.docx
       전기적 특성 테스트 (WAT)

신뢰성 시험 (HTOL, TC)

외관 검사 (Visual Inspection)

품질 개선 ...



In [36]:
# [2-6] : chunk_size 변경에 따른 분할 결과 비교
# [핵심] chunk_size 를 바꾸면 청크 수와 평균 길이가 달라짐 — 검색 품질에 직접 영향
# [주의] 실무에서는 도메인 문서 특성에 맞춰 실험적으로 최적값을 찾아야 함

for size in [200, 500, 1000]:
    splitter = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=size // 10)
    chunks = splitter.split_documents(all_docs)
    avg_len = sum(len(c.page_content) for c in chunks) / max(len(chunks), 1)
    print(f"  chunk_size={size:4d} → {len(chunks):2d}개 청크, 평균 {avg_len:.0f}자")

  chunk_size= 200 →  5개 청크, 평균 158자
  chunk_size= 500 →  3개 청크, 평균 254자
  chunk_size=1000 →  3개 청크, 평균 254자


---

## 3) 임베딩 & 벡터스토어

### 임베딩이란?

텍스트를 **고차원 벡터(숫자 배열)** 로 변환하는 과정이다. 의미가 유사한 텍스트는 벡터 공간에서 가까이 위치한다.

```
"반도체 공정" → [0.023, -0.156, 0.891, ...]  (1536차원)
"웨이퍼 제조" → [0.031, -0.142, 0.876, ...]  ← 유사한 벡터
"맛있는 김치" → [-0.512, 0.334, -0.067, ...] ← 먼 벡터
```

### 벡터스토어 비교

| 벡터스토어 | 유형 | 적합한 경우 |
|-----------|------|------------|
| **`ChromaDB`** | 로컬 파일 기반 | 개발·테스트, 소규모~중규모 데이터셋 |
| **`FAISS`** | 인프로세스 | 고성능 로컬 검색 |
| **`Pinecone`** | 매니지드 클라우드 | 프로덕션, 확장성 |

In [37]:
# [2-7] : 임베딩 동작 확인
# [핵심] embed_query — 단일 텍스트를 벡터로 변환, 결과는 float 리스트
# [주의] 임베딩 API 호출마다 비용 발생 — 대량 처리 시 embed_documents 배치 사용

# 단일 텍스트 임베딩
sample_vector = embeddings.embed_query("반도체 공정 품질 관리")

print(f"입력: '반도체 공정 품질 관리'")
print(f"벡터 차원: {len(sample_vector)}")
print(f"앞 5개 값: {sample_vector[:5]}")
print(f"타입: {type(sample_vector)}")

입력: '반도체 공정 품질 관리'
벡터 차원: 1536
앞 5개 값: [0.01554107666015625, 0.0357666015625, 0.00832366943359375, 0.0284423828125, 0.04510498046875]
타입: <class 'list'>


In [40]:
# [2-8] : 벡터스토어 구축
from langchain_core.vectorstores import InMemoryVectorStore

vectorstore = InMemoryVectorStore(embedding = embeddings)
vectorstore.add_documents(splits)  # 청크 문서 추가 — 내부적으로 임베딩 계산

['c81be383-733c-4a0a-a924-c283a5df45b3',
 '31396b25-1080-4dee-ae23-a1e2d650cc85',
 'c7b1f5ae-cb01-4cb1-b544-dbe7ae4e7100']

---

## 4) 의미론적 검색 & 검색 Tool 래핑

벡터스토어에 저장된 청크를 **질문과 의미적으로 유사한 순서**로 검색한다.

### 검색 방식

| 방식 | 메서드 | 설명 |
|------|--------|------|
| **유사도 검색** | `similarity_search(query, k)` | 코사인 유사도 기반 top-k 반환 |
| **점수 포함 검색** | `similarity_search_with_score(query, k)` | 유사도 점수와 함께 반환 |
| **MMR 검색** | `max_marginal_relevance_search(query, k)` | 관련성 + 다양성 균형 |

### Retriever 인터페이스

`vectorstore.as_retriever()` 로 **Retriever** 객체를 생성하면, LangChain 체인·에이전트에 바로 연결할 수 있다.

```python
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
docs = retriever.invoke("질문")  # similarity_search 와 동일
```

### Retriever → Agent Tool 변환

Retriever를 `@tool`로 래핑하면 **Agent가 자율적으로 검색 도구를 호출**할 수 있다. 이것이 단순 체인과 Agent 기반 RAG의 핵심 차이다.

```python
@tool
def search_documents(query: str) -> str:
    docs = retriever.invoke(query)
    return "\n".join([doc.page_content for doc in docs])
```

In [41]:
# [2-9] : similarity_search — 의미론적 유사도 검색
# [핵심] 질문을 임베딩한 뒤, 벡터 공간에서 가장 가까운 k개 청크를 반환
# [주의] k 값이 크면 노이즈 증가, 작으면 관련 정보 누락 — 보통 k=3~5
query = "반도체 품질 관리 방법은?"

# 1) 기본 유사도 검색
results = vectorstore.similarity_search(query, k=1)
results

[Document(id='c7b1f5ae-cb01-4cb1-b544-dbe7ae4e7100', metadata={'source': './sample_docs/quality_guide.docx'}, page_content='반도체 품질 관리 가이드\n\n반도체 제조 공정에서 품질 관리는 수율(Yield)을 결정하는 핵심 요소이다. 웨이퍼 단위의 결함 검사, 전기적 특성 테스트, 신뢰성 시험을 통해 제품 품질을 보증한다.\n\n주요 검사 항목\n\n결함 밀도 검사 (Defect Density)\n\n전기적 특성 테스트 (WAT)\n\n신뢰성 시험 (HTOL, TC)\n\n외관 검사 (Visual Inspection)\n\n품질 개선 프로세스\n\nPDCA(Plan-Do-Check-Act) 사이클을 기반으로 지속적 개선을 수행한다. 데이터 기반 의사결정을 위해 SPC(통계적 공정 관리) 차트를 활용하며, 이상 발생 시 8D 리포트를 작성하여 근본 원인을 분석한다.')]

In [42]:
print(f"질문: '{query}'")
print(f"검색 결과: {len(results)}개\n")

for i, doc in enumerate(results):
    source = doc.metadata.get("source", "unknown")
    print(f"  [{i+1}] (source: {source})")
    print(f"      {doc.page_content[:120]}...")
    print()

질문: '반도체 품질 관리 방법은?'
검색 결과: 1개

  [1] (source: ./sample_docs/quality_guide.docx)
      반도체 품질 관리 가이드

반도체 제조 공정에서 품질 관리는 수율(Yield)을 결정하는 핵심 요소이다. 웨이퍼 단위의 결함 검사, 전기적 특성 테스트, 신뢰성 시험을 통해 제품 품질을 보증한다.

주요 검사 항목
...



In [43]:
# [2-10] : similarity_search_with_score — 점수 확인
results_with_score = vectorstore.similarity_search_with_score(query, k=3)
print(f"질문: {query}")
print()
for i, (doc, score) in enumerate(results_with_score, 1):
    source = doc.metadata.get("source", "unknown")
    print(f"  [{i}] Score: {score:.4f} | source: {source}")
    print(f"      {doc.page_content[:80]}...")
    print()


질문: 반도체 품질 관리 방법은?

  [1] Score: 0.7798 | source: ./sample_docs/quality_guide.docx
      반도체 품질 관리 가이드

반도체 제조 공정에서 품질 관리는 수율(Yield)을 결정하는 핵심 요소이다. 웨이퍼 단위의 결함 검사, 전기적 특성...

  [2] Score: 0.2902 | source: ./sample_docs/process_data.xlsx
      공정단계 설비명 주요파라미터 정상범위 비고 노광 ASML EUV 초점거리(nm) 13.2-13.6 EUV 리소그래피 식각 Lam Research...

  [3] Score: 0.0446 | source: ./sample_docs/ai_agent_overview.pdf
      AI Agent Overview
AI agents use LLMs to understand user goals and perform tasks....



In [44]:
# [2-11] : Retriever 인터페이스 구성
# [핵심] as_retriever() — 벡터스토어를 체인·에이전트에 연결 가능한 Retriever 로 변환
# [주의] search_type="mmr" 사용 시 fetch_k (후보 수)도 함께 설정해야 효과적
# vectorstore = chroma_store  # 앞서 생성한 벡터스토어 참조
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 1})


In [45]:
# Retriever 는 invoke() 로 호출
retrieved_docs = retriever.invoke("AI 에이전트 패턴에 대해 알려줘.")
retrieved_docs

[Document(id='c81be383-733c-4a0a-a924-c283a5df45b3', metadata={'source': './sample_docs/ai_agent_overview.pdf'}, page_content='AI Agent Overview\nAI agents use LLMs to understand user goals and perform tasks.\nCore components: LLM, tools, memory.\nRepresentative patterns: ReAct, Plan-and-Execute, Reflection, RAG.')]

In [47]:
print(f"Retriever 검색 결과: {len(retrieved_docs)}개\n")
for i, doc in enumerate(retrieved_docs):
    print(f"  [{i+1}] {doc.page_content[:100]}...")
    print()

Retriever 검색 결과: 1개

  [1] AI Agent Overview
AI agents use LLMs to understand user goals and perform tasks.
Core components: LL...



---

## 5) Retriever를 Agent의 검색 Tool로 래핑

Retriever는 `invoke(query)` 로 문서를 반환하는 **검색 인터페이스**다. 그러나 Agent가 이를 직접 호출하려면 **`@tool` 데코레이터로 래핑**해야 한다.

### 왜 Tool로 래핑하는가?

| 구분 | Retriever 직접 사용 | @tool 래핑 |
|------|-------------------|-----------|
| **호출 주체** | 개발자가 코드로 호출 | Agent가 자율적으로 호출 |
| **호출 시점** | 항상 (체인에 고정) | Agent가 필요하다고 판단할 때 |
| **호출 횟수** | 1회 고정 | 필요에 따라 N회 |
| **다른 도구와 조합** | 불가 | 여러 도구와 조합 가능 |

> **핵심 전환:** `retriever.invoke(query)` → `@tool` 래핑 → Agent가 **언제, 어떤 쿼리로** 검색할지 자율 결정

In [50]:
# [2-12] : Retriever를 Agent의 검색 Tool로 래핑
# [핵심] @tool 래핑 → Agent가 자율적으로 검색 도구를 호출할 수 있는 구조
# [주의] 도구의 docstring이 Agent의 도구 선택 기준 — 명확하게 작성해야 함
from langchain_core.tools import tool

@tool
def search_documents(query: str) -> str:
    """문서에서 관련 정보를 검색합니다. 반도체 공정, 품질 관리, AI 에이전트 관련 질문에 사용하세요."""
    # [핵심] Retriever를 @tool로 래핑하면 Agent가 자율적으로 호출 가능
    docs = retriever.invoke(query)
    return "\n\n".join([doc.page_content for doc in docs])

# 도구 메타데이터 확인
print(f"도구 이름: {search_documents.name}")
print(f"도구 설명: {search_documents.description}")
print(f"도구 스키마: {search_documents.args_schema.model_json_schema()}")

도구 이름: search_documents
도구 설명: 문서에서 관련 정보를 검색합니다. 반도체 공정, 품질 관리, AI 에이전트 관련 질문에 사용하세요.
도구 스키마: {'description': '문서에서 관련 정보를 검색합니다. 반도체 공정, 품질 관리, AI 에이전트 관련 질문에 사용하세요.', 'properties': {'query': {'title': 'Query', 'type': 'string'}}, 'required': ['query'], 'title': 'search_documents', 'type': 'object'}


In [52]:
# 도구 단독 테스트
result = search_documents.invoke({"query": "AI 에이전트 아키텍처"})
print(f"\n검색 결과 미리보기:\n{result[:300]}...")


검색 결과 미리보기:
AI Agent Overview
AI agents use LLMs to understand user goals and perform tasks.
Core components: LLM, tools, memory.
Representative patterns: ReAct, Plan-and-Execute, Reflection, RAG....


---

## 6-1) Naive RAG 아키텍처 — Retrieve → Generate

### Naive RAG 파이프라인

```
사용자 질문 → [Retrieve] 벡터 검색 → [Augment] 컨텍스트 구성 → [Generate] LLM 응답
```

**Naive RAG**의 핵심은 단순함이다:
1. 질문을 임베딩하여 벡터스토어에서 관련 문서 **k개**를 검색한다
2. 검색된 문서를 **컨텍스트**로 조합한다
3. `"컨텍스트 + 질문"` 을 LLM에 전달하여 답변을 **생성**한다

### LCEL (LangChain Expression Language) 체인

LangChain의 `|` 연산자로 **Retriever → Prompt → LLM → Parser** 를 파이프라인으로 연결한다.

```python
chain = retriever | prompt | llm | output_parser
```

In [64]:
# [2-13] : Naive RAG 체인 구성 — LCEL 파이프라인
# [핵심] RunnablePassthrough 로 질문을 그대로 전달하면서, retriever 로 컨텍스트를 병렬 주입
# [주의] 프롬프트 템플릿의 {context} 와 {question} 변수명이 체인 입력 키와 일치해야 함
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# --- 1) RAG 프롬프트 템플릿 ---
rag_prompt = ChatPromptTemplate.from_template("""
다음 컨텍스트를 기반으로 질문에 답변하세요.
컨텍스트에 없는 내용은 "제공된 문서에서 해당 정보를 찾을 수 없습니다"라고 답변하세요.

컨텍스트:
{context}

질문: {question}

답변:""")

In [66]:
# --- 2) Naive RAG 체인 조립 ---
naive_rag_chain = (
    {
        "context": search_documents,    # 질문 → 검색 → 포맷팅
        "question": RunnablePassthrough(),      # 질문 그대로 전달
    }
    | rag_prompt    # 프롬프트 생성
    | llm           # LLM 호출
    | StrOutputParser()  # 문자열 파싱
)

print("Naive RAG 체인 구성 완료")
print(f"  파이프라인: retriever → format_docs → prompt → llm → str_parser")

Naive RAG 체인 구성 완료
  파이프라인: retriever → format_docs → prompt → llm → str_parser


In [67]:
# [2-14] : Naive RAG 체인 실행
# [핵심] invoke() 에 질문 문자열을 전달하면 검색 → 생성 → 답변이 자동 수행됨
# [주의] 첫 호출 시 임베딩 API + LLM API 2회 호출 — 응답 시간 수 초 소요

question = "반도체 품질 관리에서 사용하는 검사 항목은 무엇인가요?"
answer = naive_rag_chain.invoke(question)

print(f"질문: {question}\n")
print(f"답변:\n{answer}")

질문: 반도체 품질 관리에서 사용하는 검사 항목은 무엇인가요?

답변:
반도체 품질 관리에서 사용하는 검사 항목은 결함 밀도 검사(Defect Density), 전기적 특성 테스트(WAT), 신뢰성 시험(HTOL, TC), 외관 검사(Visual Inspection)입니다.


In [68]:
# [2-15] : 다양한 질문으로 Naive RAG 테스트
# [핵심] 도메인 내 질문과 도메인 외 질문을 비교하여 RAG 의 동작 확인
# [주의] 문서에 없는 정보에 대해서는 "찾을 수 없습니다" 응답이 나와야 정상

test_questions = [
    "AI 에이전트의 핵심 구성 요소는 무엇인가요?",           # PDF 문서 내용
    "반도체 식각 공정의 정상 속도 범위는?",                   # Excel 문서 내용
    "품질 개선 프로세스에서 PDCA란 무엇인가요?",             # Word 문서 내용
    "SK하이닉스의 2024년 매출은 얼마인가요?",                # 문서에 없는 질문
]

for q in test_questions:
    answer = naive_rag_chain.invoke(q)
    print(f"Q: {q}")
    print(f"A: {answer}")
    print("-" * 60)

Q: AI 에이전트의 핵심 구성 요소는 무엇인가요?
A: AI 에이전트의 핵심 구성 요소는 LLM, 도구(tools), 메모리(memory)입니다.
------------------------------------------------------------
Q: 반도체 식각 공정의 정상 속도 범위는?
A: 제공된 문서에서 해당 정보를 찾을 수 없습니다.
------------------------------------------------------------
Q: 품질 개선 프로세스에서 PDCA란 무엇인가요?
A: PDCA란 Plan(계획)-Do(실행)-Check(점검)-Act(조치)의 약자로, 품질 개선을 위해 지속적으로 반복하는 사이클을 의미합니다. 이 사이클을 통해 계획을 세우고, 실행하며, 결과를 점검하고, 필요한 조치를 취하여 공정과 품질을 지속적으로 개선합니다.
------------------------------------------------------------
Q: SK하이닉스의 2024년 매출은 얼마인가요?
A: 제공된 문서에서 해당 정보를 찾을 수 없습니다.
------------------------------------------------------------


---

## 6-2) Agent가 검색 Tool을 자율 호출하는 Naive RAG

앞서 구성한 파이프라인을 **Agent 기반 RAG** 로 확장한다. Agent는 검색 Tool을 **자율적으로 호출**하여 필요한 정보를 능동적으로 검색한다.

### LCEL 체인 vs Agent 기반 RAG

| 항목 | LCEL 체인 | Agent RAG |
|------|-------------------|-------------------|
| **검색 주체** | 코드가 고정 호출 | **Agent가 자율 판단** |
| **검색 횟수** | 1회 고정 | 필요에 따라 N회 |
| **검색 판단** | 항상 검색 | Agent가 필요 여부 판단 |
| **도구 조합** | 불가 | 여러 도구 조합 가능 |
| **미들웨어 적용** | 불가 | 하네스 미들웨어로 행동 제어 |

### Agent RAG 동작 흐름

```
사용자 질문 → Agent 판단 → [검색 Tool 호출] → 검색 결과 수신 → [LLM 생성] → 답변
                ↑                                              |
                └──── 정보 부족 시 재검색 ─────────────────────┘
```

In [59]:
# [2-16] : RAG 에이전트 생성 — create_agent
# [핵심] create_agent — 도구 호출을 자율적으로 결정하는 ReAct 에이전트 생성
# [주의] system_prompt 에 "반드시 검색 후 답변" 지침을 포함해야 환각 방지
from langchain.agents import create_agent

# 시스템 프롬프트: RAG 에이전트의 행동 규칙
RAG_SYSTEM_PROMPT = """당신은 문서 기반 Q&A 어시스턴트입니다.

규칙:
1. 질문을 받으면 반드시 search_documents 도구를 사용하여 관련 문서를 검색하세요.
2. 검색된 문서의 내용만을 기반으로 답변하세요.
3. 문서에서 답을 찾을 수 없으면 "제공된 문서에서 해당 정보를 찾을 수 없습니다"라고 솔직하게 답하세요.
4. 답변 시 출처(source)를 함께 언급하세요.
5. 한국어로 답변하세요."""

In [60]:
rag_agent = create_agent(
    model=llm,
    tools=[search_documents],
    system_prompt=RAG_SYSTEM_PROMPT,
)

In [63]:
# [2-17] : RAG 에이전트 실행 — End-to-End Q&A
# [핵심] 에이전트가 자율적으로 도구를 호출하여 검색 → 생성 수행
# [주의] 에이전트 메시지 형식은 {"messages": [...]} — LCEL 체인과 입력 형식이 다름
response = rag_agent.invoke(
    {"messages": [{"role": "user", "content": "sk하이닉스 2024년 매출은 얼마인가요?"}]}
)

# 최종 답변 출력
print("=== RAG 에이전트 응답 ===\n")
print(response["messages"][-1].content)

=== RAG 에이전트 응답 ===

제공된 문서에서 SK하이닉스 2024년 매출에 대한 정보를 찾을 수 없습니다.


In [62]:
# 에이전트 실행 흐름 확인 (메시지 타입별)
print("\n=== 실행 흐름 ===")
for msg in response["messages"]:
    role = getattr(msg, "type", "unknown")
    content_preview = str(msg.content)[:80] if msg.content else "(empty)"
    print(f"  [{role:10s}] {content_preview}...")


=== 실행 흐름 ===
  [human     ] 반도체 품질 개선 프로세스에 대해 설명해주세요....
  [ai        ] (empty)...
  [tool      ] 반도체 품질 관리 가이드

반도체 제조 공정에서 품질 관리는 수율(Yield)을 결정하는 핵심 요소이다. 웨이퍼 단위의 결함 검사, 전기적 특성...
  [ai        ] 반도체 품질 개선 프로세스는 PDCA(Plan-Do-Check-Act) 사이클을 기반으로 지속적으로 개선하는 방식입니다. 주요 단계는 다음과 같...


> **`content_and_artifact`** 반환 패턴 — 도구가 `(content, artifact)` 튜플을 반환하면, `content` 는 Agent에게 전달되고 `artifact` 는 후속 처리용 원본 데이터로 보존된다.

In [65]:
# [2-18] : 검색 도구 정의 — content_and_artifact 패턴
# [핵심] response_format="content_and_artifact" — 에이전트용 텍스트 + 원본 Document 분리
# [주의] 도구 docstring 이 에이전트의 도구 선택 기준 — 명확하게 작성해야 함
from langchain_core.tools import tool

@tool(response_format="content_and_artifact")
def retrieve_documents(query: str):
    """벡터스토어에서 관련 문서를 검색합니다. 반도체 공정, 품질 관리, AI 에이전트 관련 질문에 사용하세요."""
    docs = vectorstore.similarity_search(query, k=3)
    # content: 에이전트가 읽을 텍스트 요약
    content = "\n\n".join(
        f"[출처: {d.metadata.get('source', '?')}]\n{d.page_content}"
        for d in docs
    )
    # artifact: 후속 처리용 원본 Document 리스트
    return content, docs

# 도구 단독 테스트
test_result = retrieve_documents.invoke({"query": "품질 검사 항목"})
print("검색 도구 테스트:")
print(test_result[:200], "...")

검색 도구 테스트:
[출처: ./sample_docs/quality_guide.docx]
반도체 품질 관리 가이드

반도체 제조 공정에서 품질 관리는 수율(Yield)을 결정하는 핵심 요소이다. 웨이퍼 단위의 결함 검사, 전기적 특성 테스트, 신뢰성 시험을 통해 제품 품질을 보증한다.

주요 검사 항목

결함 밀도 검사 (Defect Density)

전기적 특성 테스트 (W ...


In [ ]:
# RAG Agent 생성
rag_agent = 

response = rag_agent.invoke(
    {"messages": [{"role": "user", "content": "반도체 품질 개선 프로세스에 대해 설명해주세요."}]}
)

# 최종 답변 출력
print("=== RAG 에이전트 응답 ===\n")
print(response["messages"][-1].content)

=== RAG 에이전트 응답 ===

반도체 품질 개선 프로세스는 다음과 같이 진행됩니다.

1. PDCA(Plan-Do-Check-Act) 사이클을 기반으로 지속적으로 개선합니다.
2. 데이터 기반 의사결정을 위해 SPC(통계적 공정 관리) 차트를 활용합니다.
3. 이상 발생 시 8D 리포트를 작성하여 근본 원인을 분석합니다.
4. 주요 검사 항목에는 결함 밀도 검사, 전기적 특성 테스트, 신뢰성 시험, 외관 검사가 포함됩니다.
5. 각 공정 단계별로 정상 범위와 주요 파라미터를 관리하여 품질을 유지합니다.

이러한 절차를 통해 반도체 제조 공정에서 제품의 품질을 보증하고, 수율을 높이기 위한 지속적인 개선이 이루어집니다.

출처: 반도체 품질 관리 가이드, 공정단계 데이터 [./sample_docs/quality_guide.docx, ./sample_docs/process_data.xlsx]


---

## 정리

| 항목 | 내용 |
|------|------|
| **다룬 기술** | 문서 로딩(PDF·Excel·Word), 텍스트 분할, 임베딩 & 벡터스토어, 의미론적 검색, **Retriever→Tool 래핑**, **미들웨어(하네스)**, Agent 기반 Naive RAG, 프롬프트 튜닝 |
| **핵심 개념** | `Document` 변환, 청킹(chunk_size/overlap), 코사인 유사도 top-k 검색, `@tool` 래핑으로 Agent 자율 호출, `SummarizationMiddleware` 컨텍스트 관리, Retrieve → Generate 파이프라인 |

### 이 Lab에서 배운 Agent 관점의 핵심 전환

| 전환 | Before | After |
|------|--------|-------|
| **검색 주체** | 코드가 `retriever.invoke()` 직접 호출 | Agent가 `@tool` 래핑된 검색 도구를 자율 호출 |
| **행동 제어** | 코드 로직으로 제어 | 미들웨어 스택(하네스)으로 선언적 제어 |
| **컨텍스트 관리** | 수동 토큰 계산 | `SummarizationMiddleware`가 자동 요약 |

### Naive RAG 한계 → Advanced RAG 전환 근거

| Naive RAG 한계 | 원인 | Advanced RAG 해결책 |
|---------------|------|-------------------|
| 단일 검색으로 정보 부족 | k개 고정 검색 | **Multi-Query** — 질문을 여러 변형으로 재작성하여 다각도 검색 |
| 키워드 불일치 | 동의어·표현 차이 | **Query Rewriting** — LLM이 검색에 최적화된 쿼리로 변환 |
| 검색 결과에 노이즈 | 관련 없는 문서 포함 | **Re-ranking** — Cross-encoder 로 검색 결과를 재정렬 |
| 긴 문서에서 핵심 누락 | 고정 크기 청킹 | **Semantic Chunking** — 의미 단위로 동적 분할 |
| 다단계 추론 불가 | 단일 생성 | **Agentic RAG** — 반복 검색 + 추론 루프 |
| 환각 검증 불가 | 사후 검증 없음 | **Self-RAG** — 생성 후 관련성·정확성 자체 평가 |